# 06 TFLite export
Export `cnn_uci.keras` as float32, dynamic-range and full int8 TFLite; compare size, test accuracy / macro F1 and latency (batch 1, 1 thread, this PC's CPU). `.tflite` files go to `models/` (gitignored).

In [1]:
import os, sys, numpy as np, tensorflow as tf
sys.path.insert(0, "../src")
from sklearn.metrics import accuracy_score, f1_score
from data_loader import load_uci
from preprocess import normalize
from export import to_tflite, predict_tflite
ROOT = os.environ.get("UCI_ROOT", "../data/uci/UCI HAR Dataset")
MODEL = os.environ.get("CNN_PATH", "../models/cnn_uci.keras")
d = load_uci(ROOT)
norm = lambda X: normalize(X, "../config/norm_stats.npz").astype("float32")
Xtr, Xte, yte = norm(d["train"]["X_raw"]), norm(d["test"]["X_raw"]), d["test"]["y"]
model = tf.keras.models.load_model(MODEL)
rep = Xtr[np.random.default_rng(0).choice(len(Xtr), 200, replace=False)]

In [2]:
rows = []
keras_pred = model.predict(Xte, verbose=0).argmax(1)
rows.append(("Keras (reference)", os.path.getsize(MODEL)/1024, accuracy_score(yte, keras_pred), f1_score(yte, keras_pred, average="macro"), None))
for name, q in [("float32", None), ("dynamic-range", "dynamic"), ("full int8", "int8")]:
    p = f"../models/cnn_uci_{name.replace(' ', '_')}.tflite"
    kb = to_tflite(model, p, q, rep) / 1024
    probs, ms = predict_tflite(p, Xte, timed=True)
    pred = probs.argmax(1)
    rows.append((name, kb, accuracy_score(yte, pred), f1_score(yte, pred, average="macro"), ms))
print("| Model | Size (KB) | Accuracy | Macro F1 | ms/window |")
print("|---|---|---|---|---|")
for n, kb, a, f, ms in rows:
    print(f"| {n} | {kb:.0f} | {a:.3f} | {f:.3f} | {'-' if ms is None else f'{ms:.3f}'} |")

INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp8nr9yw5s\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp8nr9yw5s\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmp8nr9yw5s'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 9), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1822136952848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136954768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136953616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136953424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136957072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136957840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136956880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136958800: TensorSpec(shape=(), dtype=tf.resource, name=None)


D:\Projects\SKILL LAB IBM\.venv-har\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp49be627o\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp49be627o\assets


... (4 outputs left out)

Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmpi4q424pb'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 9), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1822136952848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136954768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136953616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136953424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136957072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136957840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136956880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1822136958800: TensorSpec(shape=(), dtype=tf.resource, name=None)


D:\Projects\SKILL LAB IBM\.venv-har\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


D:\Projects\SKILL LAB IBM\.venv-har\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


| Model | Size (KB) | Accuracy | Macro F1 | ms/window |
|---|---|---|---|---|
| Keras (reference) | 4820 | 0.922 | 0.922 | - |
| float32 | 1601 | 0.922 | 0.922 | 0.658 |
| dynamic-range | 410 | 0.922 | 0.922 | 0.250 |
| full int8 | 411 | 0.923 | 0.924 | 0.123 |
